# 01 · Data Acquisition

Download all 13 data sources into `data/raw/`. Most downloads are idempotent — re-running skips files that already exist.

**Before running**: set `CENSUS_API_KEY` env var (get a free key at https://api.census.gov/data/key_signup.html).

In [1]:
import sys, os
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
pd.set_option('display.max_columns', 60)
from config import (
    DATA_PATHS, HURRICANE_META, STATES_IN_SCOPE,
    TARGET_COL, TARGET_CLASS_COL, FEATURE_GROUPS,
    RANDOM_STATE, SEVERITY_BINS, SEVERITY_LABELS,
)
RAW = DATA_PATHS['raw']; INTERIM = DATA_PATHS['interim']
PROC = DATA_PATHS['processed']; MODELS = DATA_PATHS['models']
OUT = DATA_PATHS['outputs']


In [2]:
from src.data_acquisition import download_all
download_all(census_key=os.environ.get('CENSUS_API_KEY'))


=== 1. FEMA Housing Assistance (TARGET) ===

=== 2. FEMA IHP Valid Registrations (validation only) ===

=== 3. FEMA Disaster Declarations ===
[skip] fema_disaster_declarations.csv

=== 4. Census ACS 5-year ===
[skip] census_acs5_zcta.csv

=== 5. IBTrACS North Atlantic ===
[skip] ibtracs_na.csv already exists (57,074,222 bytes)

=== 6. USDA Food Access Atlas ===
[skip] food_access_atlas.xlsx already present

=== 7. USDA SNAP Retailer Historical ===
[skip] snap_retailers.zip already present

=== 8. CDC SVI 2022 ===
[skip] cdc_svi_2022.csv already present

=== 9. HUD Tract-ZIP Crosswalk ===
[skip] hud_tract_zip.xlsx already present

=== 10. Census TIGER/Line ZCTA shapefile ===

=== 11. NOAA Storm Events (validation) ===
[skip] noaa_storm_events_2017.csv.gz already exists (9,343,148 bytes)
[skip] noaa_storm_events_2018.csv.gz already exists (9,585,982 bytes)
[skip] noaa_storm_events_2019.csv.gz already exists (11,602,648 bytes)
[skip] noaa_storm_events_2020.csv.gz already exists (10,444,6

## Sanity checks

In [3]:
for p in sorted(RAW.glob('*')):
    sz = p.stat().st_size if p.is_file() else sum(f.stat().st_size for f in p.rglob('*') if f.is_file())
    print(f'{p.name:50s}  {sz/1e6:8.2f} MB')

.gitkeep                                                0.00 MB
cdc_svi_2022.csv                                       61.10 MB
census_acs5_zcta.csv                                    2.90 MB
fema_disaster_declarations.csv                          0.14 MB
fema_housing_owners_4332.csv                            0.22 MB
fema_housing_owners_4393.csv                            0.10 MB
fema_housing_owners_4399.csv                            0.03 MB
fema_housing_owners_4559.csv                            0.08 MB
fema_housing_owners_4570.csv                            0.04 MB
fema_housing_owners_4611.csv                            0.14 MB
fema_housing_owners_4673.csv                            0.34 MB
fema_housing_renters_4332.csv                           0.20 MB
fema_housing_renters_4393.csv                           0.08 MB
fema_housing_renters_4399.csv                           0.03 MB
fema_housing_renters_4559.csv                           0.08 MB
fema_housing_renters_4570.csv           

In [4]:
# Peek at Housing Assistance for Harvey
o = pd.read_csv(RAW / 'fema_housing_owners_4332.csv')
r = pd.read_csv(RAW / 'fema_housing_renters_4332.csv')
print('owners', o.shape, '| renters', r.shape)
print(o.columns.tolist()[:20])

owners (1491, 24) | renters (1624, 21)
['disasterNumber', 'state', 'county', 'city', 'zipCode', 'validRegistrations', 'averageFemaInspectedDamage', 'totalInspected', 'totalDamage', 'noFemaInspectedDamage', 'femaInspectedDamageBetween1And10000', 'femaInspectedDamageBetween10001And20000', 'femaInspectedDamageBetween20001And30000', 'femaInspectedDamageGreaterThan30000', 'approvedForFemaAssistance', 'totalApprovedIhpAmount', 'repairReplaceAmount', 'rentalAmount', 'otherNeedsAmount', 'approvedBetween1And10000']
